In [ ]:
import pysam
import pandas as pd
import argparse
import sys
import re
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import numpy as np
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description='Extract specific fields from BAM file into a DataFrame')
    parser.add_argument('-i', '--input', required=True, help='Input BAM file')
    parser.add_argument('-o', '--output', help='Output CSV file (default: print to stdout)')
    parser.add_argument('-t', '--tag', default='GX', choices=['GX', 'GN'], 
                        help='Tag to extract feature from (GX or GN)')
    parser.add_argument('-s', '--separator', default='\t', help='Field separator for output')
    parser.add_argument('-l', '--limit', type=int, help='Limit number of rows to process')
    return parser.parse_args()

def extract_feature_from_id(sequence_id):
    """Extract the feature part from the sequence ID."""
    # Expected format: BARCODE:FEATURE:OTHER:PARTS
    parts = sequence_id.split(':')
    if len(parts) >= 2:
        # Extract just the feature name
        feature_with_number = parts[1]
        # Use regex to extract just the feature name (anything before _dup followed by digits)
        match = re.match(r'([^:]+)(?:_dup\d+)?', feature_with_number)
        if match:
            return match.group(1)
    return "Unknown"

def bam_to_dataframe(input_file, feature_tag='GX', limit=None):
    """Read BAM file and convert specified fields to a DataFrame."""
    data = []
    count = 0
    
    with pysam.AlignmentFile(input_file, "rb", threads=8) as bam:
        for read in bam:
            if limit is not None and count >= limit:
                break
                
            sequence_id = read.query_name
            mapping_quality = read.mapping_quality
            chromosome = read.reference_name
            start = read.reference_start  # 0-based inclusive
            end = read.reference_end      # 0-based exclusive

            # Extract mismatches from NM tag
            nm = read.get_tag('NM') if read.has_tag('NM') else 0
            
            # Extract local alignment score
            las = read.get_tag('AS') if read.has_tag('AS') else 0

            # Extract stellarscope posterior probabiliy score
            posterior = read.get_tag('XP') if read.has_tag('XP') else 0
            

            # Extract feature from GX or GN tag
            feature = read.get_tag(feature_tag) if read.has_tag(feature_tag) else "-"
            
            # Extract feature from sequence ID
            feature_match = re.search(r':([^:]+):', sequence_id)
            id_feature = feature_match.group(1) if feature_match else "Unknown"
            
            data.append({
                'sequenceID': sequence_id,
                'chromosome': chromosome,
                'start': start,
                'end': end,
                'feature_in_ID': id_feature,
                'quality': mapping_quality,
                f'{feature_tag}_tag': feature,
                'mismatches': nm,
                'las': las,
                'posterior': posterior
            })
            
            count += 1
    
    return pd.DataFrame(data)

def main():
    args = parse_args()
    
    try:
        df = bam_to_dataframe(args.input, args.tag, args.limit)
        
        if args.output:
            df.to_csv(args.output, sep=args.separator, index=False)
            print(f"Output written to {args.output}")
        else:
            print(df.to_csv(sep=args.separator, index=False))
            
    except Exception as e:
        sys.stderr.write(f"Error: {str(e)}\n")
        sys.exit(1)

In [ ]:
dataset_id = "simulated_mm_RA"

# SoloTE upload

In [ ]:
SoloTE_path = "" # path of SoloTE with thr 0

In [ ]:
## Import SoloTE bam with all multi-mappers (OLD simulation)
# sample = "old"
# df_SoloTE_old = bam_to_dataframe(SoloTE_path + dataset_id + "/" + sample + "/" + sample + "_SoloTE_temp/"+sample+"_teannotated.bam", 
#                     limit=None)
# gc.collect()

In [ ]:
## Import SoloTE bam with all multi-mappers (YOUNG simulation)
# sample = "young"
# df_SoloTE_young = bam_to_dataframe(SoloTE_path + dataset_id + "/" + sample + "/" + sample + "_SoloTE_temp/"+sample+"_teannotated.bam", 
#                     limit=None)
# gc.collect()

In [ ]:
#df_SoloTE_old.to_csv("data/df_SoloTE_old.csv")
df_SoloTE_old = pd.read_csv("data/df_SoloTE_old.csv")

#df_SoloTE_young.to_csv("data/df_SoloTE_young.csv")
df_SoloTE_young = pd.read_csv("data/df_SoloTE_young.csv")

# Stellarscope upload

In [ ]:
stellarscope_path = "/mnt/transfer/results/stellarscope_out/"

## Stellarscope Old

<!-- Primary alignment obtained with 

samtools view -b -F 0x100 old_pseudobulk-tmp_tele.bam > old_pseudobulk_primary_tele.bam

samtools view -h old_pseudobulk-updated.bam | awk 'BEGIN{OFS="\t"} /^@/ {print; next} $0 ~ /ZT:Z:PRI/ {print}' | samtools view -b -o old_pseudobulk_primary_updated.bam -->

Best exclude filtering of updated bam obtained in the script: filter_bam_bestExclude.sh

In [ ]:
sample = "old"

In [ ]:
## Import bam pre EM (OLD simulation)

# df_Stellarscope_old_pre = bam_to_dataframe(stellarscope_path + dataset_id + "_pseudobulk_onestep/" + sample + "/" + sample + "_pseudobulk_primary_tele.bam", 
#                                         feature_tag='ZF', limit=None)
# gc.collect()
# df_Stellarscope_old_pre[df_Stellarscope_old_pre["sequenceID"].duplicated()] # no reads are repeated

In [ ]:
## Import bam post EM (OLD simulation)
# sample = "old"
# df_Stellarscope_old_updated = bam_to_dataframe(stellarscope_path + dataset_id + "_pseudobulk_onestep/" + sample + "/" + sample + "_pseudobulk_primary_updated.bam", 
#                                         feature_tag='ZF', limit=None)
# gc.collect()

In [ ]:
## Add column with original MAPQ (to separate unique and multi-mapping reads)
# df_Stellarscope_old_updated['MAPQ_Pre'] = df_Stellarscope_old_updated['sequenceID'].map(df_Stellarscope_old_pre.set_index('sequenceID')['quality'])


In [ ]:
# df_Stellarscope_old_updated.MAPQ_Pre.value_counts()
# df_Stellarscope_old_updated.quality.value_counts()

In [ ]:
# df_Stellarscope_old_pre.to_csv("data/df_Stellarscope_old_pre.csv")
# df_Stellarscope_old_updated.to_csv("data/df_Stellarscope_old_updated.csv")


In [ ]:
# bam before EM
df_Stellarscope_old_pre = pd.read_csv("data/df_Stellarscope_old_pre.csv")

In [ ]:
# # bam after EM, bestExclude strategy
# df_Stellarscope_old_bestExclude = bam_to_dataframe(stellarscope_path + dataset_id + \
#     "_pseudobulk_onestep/" + sample + "/" + sample + "_pseudobulk_bestExclude_vScript.bam", \
#     feature_tag='ZF', limit=None)

# df_Stellarscope_old_bestExclude['MAPQ_Pre'] = df_Stellarscope_old_bestExclude['sequenceID'].map(df_Stellarscope_old_pre.set_index('sequenceID')['quality'])

# df_Stellarscope_old_bestExclude.shape

# gc.collect()

In [ ]:
#df_Stellarscope_old_bestExclude.to_csv("data/df_Stellarscope_old_bestExclude.csv")
df_Stellarscope_old_bestExclude = pd.read_csv("data/df_Stellarscope_old_bestExclude.csv")

In [ ]:
# df_Stellarscope_old_updated_05 = bam_to_dataframe(stellarscope_path + dataset_id + \
#     "_pseudobulk_onestep/" + sample + "/" + sample + "_pseudobulk_posterior_gt_0.5.bam", \
#     feature_tag='ZF', limit=None)

# df_Stellarscope_old_updated_05['MAPQ_Pre'] = df_Stellarscope_old_updated_05['sequenceID'].map(df_Stellarscope_old_pre.set_index('sequenceID')['quality'])

# df_Stellarscope_old_updated_05.shape

In [ ]:
# df_Stellarscope_old_updated_05.to_csv("data/df_Stellarscope_old_updated_05.csv")

### Filter by posterior

#### 0.5

In [ ]:
df_Stellarscope_old_updated_05 = df_Stellarscope_old_bestExclude.loc[df_Stellarscope_old_bestExclude.posterior > 0.5,:].copy()
df_Stellarscope_old_updated_05.shape

#### 0.9

In [ ]:
df_Stellarscope_old_updated_09 = df_Stellarscope_old_updated_05.loc[df_Stellarscope_old_updated_05.posterior > 0.9,:].copy()
df_Stellarscope_old_updated_09

In [ ]:
df_Stellarscope_old_updated_095 = df_Stellarscope_old_updated_09.loc[df_Stellarscope_old_updated_09.posterior > 0.95,:].copy()
df_Stellarscope_old_updated_095

#### 0.99

In [ ]:
df_Stellarscope_old_updated_099 = df_Stellarscope_old_updated_095.loc[df_Stellarscope_old_updated_095.posterior > 0.99,:].copy()
df_Stellarscope_old_updated_099

In [ ]:
df_Stellarscope_old_pre.quality.value_counts()
df_Stellarscope_old_bestExclude.quality.describe()
df_Stellarscope_old_bestExclude.quality.value_counts()

In [ ]:
df_Stellarscope_old_pre.posterior.value_counts()
df_Stellarscope_old_bestExclude.posterior.describe()
df_Stellarscope_old_bestExclude.posterior.value_counts()

## Stellarscope Young

Get primary mapping and filter out reads with posterior probability 0 

```bash 
samtools view -@ 8 -h young_pseudobulk_primary_updated.bam \
| awk 'BEGIN{OFS="\t"} /^@/ {print; next} $0 !~ /XP:f:0(\.0*)?(\s|$)/ {print}' \
| samtools view -@ 8 -b -o young_pseudobulk_nonZeroXP_primary_updated.bam
```

In [ ]:
gc.collect()
dataset_id = "simulated_mm_RA"
sample = "young"

In [ ]:
# df_Stellarscope_young_pre = bam_to_dataframe("/mnt/transfer/results/stellarscope_out/simulated_mm_RA_pseudobulk_onestep/young/young_pseudobulk_primary_tele.bam", 
#                                         feature_tag='ZF')
# gc.collect()

In [ ]:
#df_Stellarscope_young_pre.to_csv("data/df_Stellarscope_young_pre.csv")
df_Stellarscope_young_pre = pd.read_csv("data/df_Stellarscope_young_pre.csv")


In [ ]:
df_Stellarscope_young_pre.shape

In [ ]:
# sample = "young"

# df_Stellarscope_young_bestExclude = bam_to_dataframe(stellarscope_path + dataset_id + "_pseudobulk_onestep/" + \
#     sample + "/" + sample + "_pseudobulk_bestExclude.bam", 
#                                         feature_tag='ZF')
# gc.collect()

In [ ]:
# df_Stellarscope_young_bestExclude['MAPQ_Pre'] = df_Stellarscope_young_bestExclude['sequenceID'].map(df_Stellarscope_young_pre.set_index('sequenceID')['quality'])

# df_Stellarscope_young_bestExclude.shape

In [ ]:
#df_Stellarscope_young_bestExclude.to_csv("data/df_Stellarscope_young_bestExclude.csv")
df_Stellarscope_young_bestExclude = pd.read_csv("data/df_Stellarscope_young_bestExclude.csv")

In [ ]:
df_Stellarscope_young_bestExclude.shape
df_Stellarscope_young_bestExclude.posterior.describe()
df_Stellarscope_young_bestExclude.MAPQ_Pre.value_counts()

In [ ]:
# Each sequenceID should appear exactly once
assert df_Stellarscope_young_bestExclude['sequenceID'].nunique() == len(df_Stellarscope_young_bestExclude)

### Filter by posterior

#### 0.5

In [ ]:
df_Stellarscope_young_updated_05 = df_Stellarscope_young_bestExclude.loc[df_Stellarscope_young_bestExclude.posterior > 0.5, :].copy()
df_Stellarscope_young_updated_05.shape

In [ ]:

df_Stellarscope_young_updated_05.posterior.describe()
df_Stellarscope_young_updated_05.MAPQ_Pre.value_counts()


#### 0.9

In [ ]:
df_Stellarscope_young_updated_09 = df_Stellarscope_young_updated_05.loc[df_Stellarscope_young_updated_05.posterior > 0.9, :].copy()

#### 0.95

In [ ]:
df_Stellarscope_young_updated_095 = df_Stellarscope_young_updated_09.loc[df_Stellarscope_young_updated_09.posterior > 0.95,:].copy()
df_Stellarscope_young_updated_095

#### 0.99

In [ ]:
df_Stellarscope_young_updated_099 = df_Stellarscope_young_updated_09.loc[df_Stellarscope_young_updated_09.posterior > 0.99,:].copy()
df_Stellarscope_young_updated_099.shape

# Comparison

In [ ]:
# conversion table
conversion_table = pd.read_csv("/mnt/transfer/TEbenchmarking/Rscripts/annotation/annotation_mm10_conversion_withAge.tsv", 
                        sep=" ")
conversion_table.head

## Check if the mapped feature is correct

### SoloTE

In [ ]:
# # check that the mapped feature is correct 

# SoloTE
# check that all the names match
df_SoloTE_old['GX_tag'].isin(conversion_table.soloTE_name).value_counts()
# convert SoloTE id in locus id
# Create a mapping from df2
id_to_soloTE = conversion_table.dropna().drop_duplicates('soloTE_name', keep='last').set_index('soloTE_name')['locusID']
id_to_soloTE
# Map to df1
df_SoloTE_old['locusID'] = df_SoloTE_old['GX_tag'].map(id_to_soloTE)
df_SoloTE_old['correct_feature'] = (df_SoloTE_old.feature_in_ID == df_SoloTE_old.locusID)
df_SoloTE_old['correct_feature'].value_counts()

# SoloTE
# check that all the names match
df_SoloTE_young['GX_tag'].isin(conversion_table.soloTE_name).value_counts()
# convert SoloTE id in locus id
# Create a mapping from df2
id_to_soloTE = conversion_table.dropna().drop_duplicates('soloTE_name', keep='last').set_index('soloTE_name')['locusID']
id_to_soloTE
# Map to df1
df_SoloTE_young['locusID'] = df_SoloTE_young['GX_tag'].map(id_to_soloTE)
df_SoloTE_young['correct_feature'] = (df_SoloTE_young.feature_in_ID == df_SoloTE_young.locusID)
df_SoloTE_young['correct_feature'].value_counts()


### Stellarscope

In [ ]:
# Define all dataframes to process
dataframes = {
    'Stellarscope_old_pre': df_Stellarscope_old_pre,
    'Stellarscope_old_bestExclude': df_Stellarscope_old_bestExclude,
    'Stellarscope_old_updated_05': df_Stellarscope_old_updated_05,
    'Stellarscope_old_updated_09': df_Stellarscope_old_updated_09,
    'Stellarscope_old_updated_095': df_Stellarscope_old_updated_095,
    'Stellarscope_old_updated_099': df_Stellarscope_old_updated_099,
    'Stellarscope_young_pre': df_Stellarscope_young_pre,
    'Stellarscope_young_bestExclude': df_Stellarscope_young_bestExclude,
    'Stellarscope_young_updated_05': df_Stellarscope_young_updated_05,
    'Stellarscope_young_updated_09': df_Stellarscope_young_updated_09,
    'Stellarscope_young_updated_095': df_Stellarscope_young_updated_095,
    'Stellarscope_young_updated_099': df_Stellarscope_young_updated_099,
}

# Process all dataframes
for name, df in dataframes.items():
    # Use np.where for conditional assignment
    df['correct_feature'] = np.where(
        df['ZF_tag'] == "-",
        "Unmapped",
        df['feature_in_ID'] == df['ZF_tag']
    )
    
    # Print results
    print(f"\n{name}:")
    print(df['correct_feature'].value_counts())

In [ ]:
# Define dataframes with their configurations
datasets = [
    {'df': df_SoloTE_old, 'tool': 'oldTEs_SoloTE', 'quality_col': 'quality'},
    {'df': df_SoloTE_young, 'tool': 'youngTEs_SoloTE', 'quality_col': 'quality'},
    {'df': df_Stellarscope_old_pre, 'tool': 'oldTEs_Stellarscope_PreEM', 'quality_col': 'quality'},
    {'df': df_Stellarscope_old_bestExclude, 'tool': 'oldTEs_Stellarscope_postEM_bestExclude', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_old_updated_05, 'tool': 'oldTEs_Stellarscope_postEM_thr05', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_old_updated_09, 'tool': 'oldTEs_Stellarscope_postEM_thr09', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_old_updated_095, 'tool': 'oldTEs_Stellarscope_postEM_thr095', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_old_updated_099, 'tool': 'oldTEs_Stellarscope_postEM_thr099', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_young_pre, 'tool': 'youngTEs_Stellarscope_PreEM', 'quality_col': 'quality'},
    {'df': df_Stellarscope_young_bestExclude, 'tool': 'youngTEs_Stellarscope_postEM_bestExclude', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_young_updated_05, 'tool': 'youngTEs_Stellarscope_postEM_thr05', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_young_updated_09, 'tool': 'youngTEs_Stellarscope_postEM_thr09', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_young_updated_095, 'tool': 'youngTEs_Stellarscope_postEM_thr095', 'quality_col': 'MAPQ_Pre'},
    {'df': df_Stellarscope_young_updated_099, 'tool': 'youngTEs_Stellarscope_postEM_thr099', 'quality_col': 'MAPQ_Pre'},
]

# Process all datasets
data_list = []

for config in datasets:
    # Create crosstab
    data = pd.crosstab(config['df'].correct_feature, config['df'][config['quality_col']])
    
    # Reshape and add metadata
    data = (data.reset_index()
            .melt(id_vars='correct_feature', var_name='MappingQuality', value_name='Count')
            .rename(columns={'correct_feature': 'MappingResult'})
            .assign(Tool=config['tool'])
                        .assign(Tool=config['tool'],
                    Unique=lambda x: x['MappingQuality'] == 255))
    
    data_list.append(data)

# Combine all data
data_all = pd.concat(data_list, ignore_index=True)

# Convert MappingResult to consistent boolean type
data_all['MappingResult'] = data_all['MappingResult'].map({
    True: True,
    False: False,
    'True': True,
    'False': False
})

print(f"Total rows: {len(data_all)}")
print(f"Tools: {data_all['Tool'].nunique()}")
data_all.head()

In [ ]:
data_all.to_csv("data/df_forPlotting_mouseRAsimulation_SoloTEvsStellarscopeWthresholds.csv")

# checkpoint before plotting

In [ ]:
data_all = pd.read_csv("data/df_forPlotting_mouseRAsimulation_SoloTEvsStellarscopeWthresholds.csv")

In [ ]:
data_all

### All tools and ages

In [ ]:
sns.set_context("paper")

pivot = (data_all
         .groupby(['Tool', 'MappingResult'])['Count']
         .sum()
         .reset_index()
         .pivot(index='Tool', columns='MappingResult', values='Count')
         .fillna(0)
         .rename(columns={True: 'Correct', False: 'Wrong'})
         [['Correct', 'Wrong']])

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

pivot.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=['#6dbc90', '#565656'],
    edgecolor='white',
    linewidth=0.1,
    legend=False  # Disable default legend
)

ax.set_ylabel('Count', fontsize=16)
ax.set_xlabel('Tool', fontsize=16)
ax.set_title('Mapping Results by Tool', fontsize=16, pad=20)

# Add legend to the right
ax.legend(
    ['Correct', 'Wrong'],
    title='Mapping Result',
    loc='center left',
    bbox_to_anchor=(1.0, 0.5),
    frameon=True
)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig('figures/mapping_results_by_tool_withBestExclude.pdf', 
            dpi=600, bbox_inches='tight')
plt.show()

### Old simulation

In [ ]:
sns.set_context("paper", font_scale=1.2)
# Filter for "young" tools
df = data_all[data_all['Tool'].str.startswith("old")]
# Step 1 — aggregate counts
grouped = (
    df.groupby(["Tool", "Unique", "MappingResult"], as_index=False)["Count"]
    .sum()
)

# Step 2 — pivot for plotting (raw counts)
pivot = grouped.pivot_table(
    index=["Tool", "Unique"],
    columns="MappingResult",
    values="Count"
)

# Step 3 — facet plotting
facet_order = [True, False]  # your desired facet order
name_map = {False: "Multi", True: "Uniquely"}

fig, axes = plt.subplots(1, 2, figsize=(8, 5.5), sharey=False)

for ax, unique_val in zip(axes, facet_order):
    subdf = pivot.xs(unique_val, level="Unique")
    subdf.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=['#646E78', '#84C286'],
        edgecolor="white",
        linewidth=0.1,
        legend=False  # suppress legends inside each subplot
    )
    ax.set_title(f"{name_map[unique_val]}-mapped reads")
    ax.set_ylabel("Read count")
    ax.set_xlabel("Tool")

    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')

# Create a single shared legend on the right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, ["Wrong", "Correct"],
    title="Primary mapping",
    loc="center left",
    bbox_to_anchor=(0.85, 0.69),  # right side, vertically centered
)

plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space for the legend
plt.savefig("figures/mapping_results_comparison_facetMAPQ_counts_old_withBestExclude.pdf",
            dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
sns.set_context("paper", font_scale=1.2)

# Step 1 — aggregate counts
grouped = (
    df.groupby(["Tool", "Unique", "MappingResult"], as_index=False)["Count"]
    .sum()
)

# Step 2 — normalize to percentage per Tool+Unique
grouped["Percentage"] = (
    grouped.groupby(["Tool", "Unique"])["Count"]
    .transform(lambda x: 100 * x / x.sum())
)

# Step 3 — pivot for plotting
pivot = grouped.pivot_table(
    index=["Tool", "Unique"],
    columns="MappingResult",
    values="Percentage"
)

# Step 4 — facet plotting
facet_order = [False, True]
name_map = {False: "Multi", True: "Uniquely"}

fig, axes = plt.subplots(1, 2, figsize=(7.5, 5.5), sharey=True)

for ax, unique_val in zip(axes, facet_order):
    subdf = pivot.xs(unique_val, level="Unique")
    subdf.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=['#646E78', '#84C286'],
        edgecolor="white",
        linewidth=0.1
    )
    ax.set_title(f"{name_map[unique_val]}-mapped reads")
    ax.set_ylabel("Percentage of reads")
    ax.set_xlabel("Tool")

    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')

    ax.legend_.remove()  # remove legend from each subplot

# Add single legend to the right of the figure
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, ["Wrong", "Correct"],
    title="Primary mapping",
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)

plt.tight_layout()
plt.savefig("figures/mapping_results_comparison_facetMAPQ_percentage_withBestExclude.pdf", dpi=600, bbox_inches="tight")
plt.show()

### Young

In [ ]:
sns.set_context("paper", font_scale=1.2)

# Filter for "young" tools
df = data_all[data_all['Tool'].str.startswith("young")]
df["Tool"] = df["Tool"].str.removeprefix("youngTEs_")
# Step 1 — aggregate counts
grouped = (
    df.groupby(["Tool", "Unique", "MappingResult"], as_index=False)["Count"]
    .sum()
)

# Step 2 — pivot for plotting (raw counts)
pivot = grouped.pivot_table(
    index=["Tool", "Unique"],
    columns="MappingResult",
    values="Count"
)

# Step 3 — facet plotting
facet_order = [True, False]  # your desired facet order
name_map = {False: "Multi", True: "Uniquely"}

fig, axes = plt.subplots(1, 2, figsize=(7, 5), sharey=False)

for ax, unique_val in zip(axes, facet_order):
    subdf = pivot.xs(unique_val, level="Unique")
    subdf.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=['#646E78', '#84C286'],
        edgecolor="white",
        linewidth=0.1,
        legend=False  # suppress legends inside each subplot
    )
    ax.set_title(f"{name_map[unique_val]}-mapped reads")
    ax.set_ylabel("Read count")
    ax.set_xlabel("Tool")

    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')

# Create a single shared legend on the right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, ["Wrong", "Correct"],
    title="Primary mapping",
    loc="center left",
    bbox_to_anchor=(0.85, 0.69),  # right side, vertically centered
)

plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space for the legend
plt.savefig("figures/mapping_results_comparison_facetMAPQ_counts_young_withBestExclude.pdf",
            dpi=600, bbox_inches="tight")
plt.show()
